# Lesson 3, Part 3

Here are helper functions carried forward from earlier parts.

In [ ]:
def cubic(x, a, b, c, d):
    if type(x) == int or type(x) == float: return (a * x * x + b * x + c) * x + d
    else: return [(a * val * val + b * val + c) * val + d for val in x]

def approx_deriv_cubic(x_val, a, b, c, d, nudge=0.001):
    y_minus = cubic(x_val - nudge / 2, a, b, c, d)
    y_plus = cubic(x_val + nudge / 2, a, b, c, d)
    return (y_plus - y_minus) / nudge

def linear_spacing(start, stop, num):
    interval = (stop - start) / (num - 1)
    return [start + i * interval for i in range(num)]


From the text, *x*<sup>3</sup> has derivative 3*x*<sup>2</sup>, *x*<sup>2</sup> has derivative 2*x*, *x* has derivative 1, and 1 has derivative 0. 
Multiplying any function by a constant *c* multiplies its derivative by *c*. 
The derivative of any sum of functions equals the sum of its derivatives. The combination implies the simple formula presented here.

In [ ]:
def exact_deriv_cubic(x,a,b,c,d=0):   # constant d doesn't matter, so default to 0
    if type(x)==int or type(x)==float:  
        return 3*a*x*x + 2*b*x + c
    else:
        return [3*a*val*val + 2*b*val + c for val in x]

x = [1,2,3]
print(exact_deriv_cubic(x,0,1,0))
print(exact_deriv_cubic(x,1,0,0))
print(exact_deriv_cubic(x,1,1,0))

Let's compare the exact answers with our slope approximations 
and measure the average absolute error relative to the average absolute slopes.

In [ ]:
def approx_deriv_cubic(x,a,b,c,d=0,nudge=0.001):
    deriv = []
    for val in x:
        y_minus = cubic(val-nudge/2,a,b,c,d)
        y_plus = cubic(val+nudge/2,a,b,c,d)
        slope =  (y_plus - y_minus) / nudge
        deriv.append(slope)
    return deriv

def avg_rel_abs_error(approx,exact):
    abs_error = [abs(approx[i] - exact[i]) for i in range(len(approx))]
    abs_val = [abs(val) for val in exact]
    err = sum(abs_error) / sum(abs_val)
    # err = sum(abs_error) / sum(abs_val)  + 1E-16)  # safer version
    return err

def compare_deriv_cubic(xmin,xmax,a,b,c,d=0,nudge=0.001):
    x = linear_spacing(xmin,xmax,500)
    exact = exact_deriv_cubic(x,a,b,c,d)
    approx = approx_deriv_cubic(x,a,b,c,d,nudge)
    return avg_rel_abs_error(approx,exact)
    
xmin,xmax = 0,400
a,b,c = 1,-2,100
avg_rel_abs_err = compare_deriv_cubic(xmin,xmax,a,b,c)
print('average relative absolute error =',avg_rel_abs_err)

Let's  measure the average absolute error relative 
to the average absolute slopes more systematically.

In [ ]:
xmin,xmax = 0,400
iter = sum_err = 0
for a in range(-3,4):                   # iterates though all a
    for b in range(-20,25,5):           # iterates through all b for given a 
        for c in range(-200,250,100):   # iterates through all c for given a and b
            if a==b==c==0:   # skips the case where all answers are zero 
                continue     # move forward to the next c 
            iter += 1
            sum_err += compare_deriv_cubic(xmin,xmax,a,b,c)
print('Average relative absolute error',round(sum_err/iter,12),'in',iter,'iterations')

Applying the Fundamental Theorem, we can invert exact_deriv_cubic into an exact integral of a quadratic, apart from an arbitrary constant. 
To maintain consistency in parameters, we will treat the quadratic as a cubic with multipler 0 on its *x*<sup>3</sup> term.

In [ ]:
def exact_integral_quadratic(x,b,c,d,constant=0):
    if type(x)==int or type(x)==float:  
        return (1/3)*b*x*x*x + (1/2)*c*x*x + d * x + constant
    else:
        return [(1/3)*b*val*val*val + (1/2)*c*val*val + d * val + constant for val in x]

x = [1,2,3]
print(exact_integral_quadratic(x,0,0,1))
print(exact_integral_quadratic(x,3,0,0))
print(exact_integral_quadratic(x,3,0,1))

Let's compare the exact answers with our area approximations 
and measure the average absolute relative errors.

In [ ]:
def approx_integral_quadratic(x,b,c,d):
    y = cubic(x,0,b,c,d)
    integral_y = [0]
    area_mid = 0
    for i in range(len(y)-1):
        area_mid += 0.5 * (y[i] + y[i+1]) * (x[i+1] - x[i])
        integral_y.append(area_mid)
    return integral_y

def compare_integral_quadratic(xmin,xmax,b,c,d,num_slice=500):  
    x = linear_spacing(xmin,xmax,num_slice+1)
    exact = exact_integral_quadratic(x,b,c,d)
    exact = [val - exact[0] for val in exact] # shift exact[0] to zero
    approx = approx_integral_quadratic(x,b,c,d)
    return avg_rel_abs_error(approx,exact)
    
xmin,xmax = 0,400
b,c,d = 3,-4,100
for num_slice in (500,1000,10000):
    rel_error = compare_integral_quadratic(xmin,xmax,b,c,d,num_slice)
    print(num_slice,'slices:',round(rel_error,11),'average relative absolute error')

While the average relative errors are small, they are much higher than for derivatives. 
Let's confirm this for a larger sample.

In [ ]:
xmin,xmax = 0,400
iter = sum_err = 0
for a in range(-3,4):                   # iterates though all a
    for b in range(-20,25,5):           # iterates through all b for given a 
        for c in range(-200,250,100):   # iterates through all c for given a and b
            if a==b==c==0:   # skips the case where all answers are zero 
                continue     # move forward to the next c 
            iter += 1
            sum_err += compare_integral_quadratic(xmin,xmax,a,b,c,1000)
print('Average relative absolute error',round(sum_err/iter,12),'in',iter,'iterations')

The quadratics' areas are estimated less precisely than the cubics' slopes 
mainly because their nudges are wider, 
and the nudges were set wider because much more computation is required. 
Fortunately, the exact formulas are far simpler than either. 
Let's now do a similar comparison for sqrt(*x*), whose exact derivative is 1/2/sqrt(*x*).

In [ ]:
from math import sqrt

def exact_deriv_sqrt(x_val):
    return 1 / (2 * sqrt(x_val)) 

def approx_deriv_sqrt(x_val,nudge):
    y_minus = sqrt(x_val - nudge/2)
    y_plus = sqrt(x_val + nudge/2)
    return (y_plus - y_minus) / nudge

def compare_deriv_sqrt(xmin,xmax,nudge=0.001):
    xmin = max(xmin, 0.6*nudge)  # keeps sqrt(x) and slope well-defined
    x = linear_spacing(xmin,xmax,500)
    exact = [exact_deriv_sqrt(val) for val in x]
    approx = [approx_deriv_sqrt(val,nudge) for val in x]
    return avg_rel_abs_error(approx,exact)

xmin,xmax = 1,100
rel_error  = compare_deriv_sqrt(xmin,xmax)
print('Average relative absolute error =',rel_error)

While the approximation is excellent for *x*>1, its accuracy deterioriates as *x* approaches zero.
That's because the exact derivative changes rapidly near the origin: e.g., 11.2 at *x* = 0.002 versus 15.8 at *x* = 0.001. 
While finer nudges can improve accuracy until rounding errors interfere, the exact formula is simpler and more reliable.

In [ ]:
# investigate impact of nudge
xmin,xmax = 0,0.01
for nudge in (1E-3,1E-4,1E-5,1E-6):
    rel_error = compare_deriv_sqrt(xmin,xmax,nudge)
    print('nudge',nudge,', average relative absolute error',round(rel_error,7))

Next let's evaluate slope approximations for a unit circle. 
Recall that the true slope at (*x,y*) is -*x/y*,
so that the tangent lies at right angles to a radial line from the center.

In [ ]:
def unit_circle_y(x_val,is_negative=False):
    y_val = sqrt(1 - x_val * x_val)
    if is_negative: y_val = -y_val
    return y_val 

def exact_deriv_circle(x_val):   # formula applies to any circle
    return -x_val / unit_circle_y(x_val)

def approx_deriv_unit_circle(x_val,nudge):
    y_minus = unit_circle_y(x_val - nudge/2)
    y_plus = unit_circle_y(x_val + nudge/2)
    return (y_plus - y_minus) / nudge

def compare_deriv_unit_circle(xmin,xmax,nudge=0.001,num_slice=500):
    xmax = min(abs(xmax), 1-0.6*nudge)  # keeps slope well-defined
    x = linear_spacing(xmin,xmax,num_slice+1)
    exact = [exact_deriv_circle(val) for val in x]
    approx = [approx_deriv_unit_circle(val,nudge) for val in x]
    return avg_rel_abs_error(approx,exact)

xmin,xmax = 0,0.9
rel_err  = compare_deriv_unit_circle(xmin,xmax)
print('Average relative absolute error =',round(rel_err,11))

Again, the approximation is generally excellent 
but deteriorates where the slope steepens or flattens rapidly for small nudges.

In [ ]:
xmin,xmax = 0.99,1
for nudge in (1E-3,1E-4,1E-5,1E-6):
    rel_error = compare_deriv_unit_circle(xmin,xmax,nudge)
    print('nudge',nudge,', average relative absolute error',round(rel_error,4))

The results are so similar for an ellipse that we won't explore them here. 
Instead, let's look at ways to combine the previous comparisons. 
Here is a first attempt.

In [ ]:
def compare_deriv(which_type,xmin,xmax,a,b,c):
    if which_type == 'cubic':
        return compare_deriv_cubic(xmin,xmax,a,b,c)
    else:
        if which_type == 'square root':
            return compare_deriv_sqrt(xmin,xmax)
        else:
            if which_type == 'unit circle':
                return compare_deriv_unit_circle(xmin,xmax)
            else:
                return "Choices are 'cubic', 'square root', or 'unit circle'"

xmin,xmax = 0,1
a,b,c = -100,200,300
for which_type in ('cubic','square root','unit circle','anything else'):
    print(compare_deriv(which_type,xmin,xmax,a,b,c))

While this works, the successive indentations get cumbersome. 
To simplify, we can use the Python statement **elif**, 
which stands for "else if" and lets us compress two steps into one.

In [ ]:
def compare_deriv1(which_type,xmin,xmax,a,b,c):
    if which_type == 'cubic':
        return compare_deriv_cubic(xmin,xmax,a,b,c)
    elif which_type == 'square root':
        return compare_deriv_sqrt(xmin,xmax)
    elif which_type == 'unit circle':
        return compare_deriv_unit_circle(xmin,xmax)
    else:
        return "Choices are 'cubic', 'square root', or 'unit circle'"

xmin,xmax = 0,1
a,b,c = -100,200,300
for which_type in ('cubic','square root','unit circle','anything else'):
    print(compare_deriv1(which_type,xmin,xmax,a,b,c))

Unlike **if**, **elif** is checked only if the preceding 
**if** or **elif** condition is evaluated as false. 
To make sure that it is checked, use **if** instead. 
How might you rearrange this code for better clarity?

In [ ]:
def weird_elif(x):
    if x > 0:                # check this first
        answer = 'positive'  # do this if x>0 when checked
        if x % 2 == 0:       # isnt's checked for x<=0   
            answer += ' and an even number'
        elif x > 4:
            answer += ' and greater than 4'
    elif x == 0:
        answer = 'zero' 
    elif x == 5:             # this condition will never be checked
        answer = 'five'      # hence this will never be done
    else:
        answer = 'negative'
    if x == -5:
        answer = 'minus five' 
    return answer

for x in (0,3,5,6,-6,-5):
    print(x,'is',weird_elif(x))